# Chapter 27 Coding Exam — Neural Networks from Scratch in NumPy (100 Points)

**Book Reference:** Chapter 27, Section 27.7 (Chapter 27 Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Pure-NumPy Deep Learning Workflow | Points |
| :--- | :--- | :---: |
| **Task 27.1** | `solve_xor_mlp_numpy(seed=42, epochs=2500, lr=0.5)` (2 -> 8 -> 1 XOR MLP) | 30 |
| **Task 27.2** | `train_numpy_mlp_classifier(...)` (He-init 2-layer ReLU MLP trainer) | 35 |
| **Task 27.3** | `compare_numpy_mlp_vs_logistic_mch(mch_df, seed=42)` (MCH stunting benchmark) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 27 Exam Grader
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

tutor.activate(chapter="ch27", notebook="coding_exam")


---
## Exam Task 27.1 (30 Points): `solve_xor_mlp_numpy(seed=42, epochs=2500, lr=0.5)`

Train a pure-NumPy 2-layer MLP (`2 -> 8 -> 1`, `tanh` or `ReLU` hidden layer, sigmoid output) using `np.random.default_rng(seed)` on XOR (`[[0,0],[0,1],[1,0],[1,1]] -> [0,1,1,0]`):
1. Run `epochs` full-batch gradient-descent steps minimizing mean binary cross-entropy.
2. Return `{"accuracy": float, "final_loss": round5, "preds": list_of_4_ints}` achieving `accuracy == 1.0`, `preds == [0, 1, 1, 0]`, and `final_loss < 0.1`.

In [ ]:
def solve_xor_mlp_numpy(seed=42, epochs=2500, lr=0.5):
    """Train a pure-NumPy 2->8->1 MLP on XOR and return metrics."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    rng = np.random.default_rng(seed)
    X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
    y = np.array([[0.], [1.], [1.], [0.]])
    W1 = rng.normal(0.0, np.sqrt(1.0 / 2), size=(2, 8))
    b1 = np.zeros((1, 8), dtype=float)
    W2 = rng.normal(0.0, np.sqrt(1.0 / 8), size=(8, 1))
    b2 = np.zeros((1, 1), dtype=float)
    n = len(X)
    for _ in range(epochs):
        A1 = np.tanh(X @ W1 + b1)
        Z2 = A1 @ W2 + b2
        probs = 1.0 / (1.0 + np.exp(-np.clip(Z2, -500.0, 500.0)))
        dZ2 = (probs - y) / n
        dW2 = A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dZ1 = (dZ2 @ W2.T) * (1.0 - A1 ** 2)
        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        W1 = W1 - lr * dW1
        b1 = b1 - lr * db1
        W2 = W2 - lr * dW2
        b2 = b2 - lr * db2
    A1 = np.tanh(X @ W1 + b1)
    p = 1.0 / (1.0 + np.exp(-np.clip(A1 @ W2 + b2, -500.0, 500.0)))
    pc = np.clip(p, 1e-12, 1.0 - 1e-12)
    loss = -float(np.mean(
        y * np.log(pc) + (1.0 - y) * np.log(1.0 - pc)
    ))
    preds = [int(v >= 0.5) for v in p.ravel()]
    acc = float(np.mean(np.array(preds) == y.ravel()))
    return {
        "accuracy": acc,
        "final_loss": round(loss, 5),
        "preds": preds,
    }
    ### END SOLUTION


---
## Exam Task 27.2 (35 Points): `train_numpy_mlp_classifier(X_train, y_train, X_val, y_val, hidden_dim=16, lr=0.1, epochs=300, seed=42)`

Train a pure-NumPy 2-layer ReLU MLP classifier without mutating caller arrays:
1. Initialize weights with He scaling (`rng = np.random.default_rng(seed)`, `std = np.sqrt(2.0 / d_in)` for `W1`, `np.sqrt(2.0 / hidden_dim)` for `W2`) and zero biases (`b1 (1, hidden_dim)`, `b2 (1, 1)`).
2. Track mean BCE `train_losses` and `val_losses` across all `epochs`, updating parameters strictly from training gradients.
3. Compute `val_roc_auc` rounded to **4 decimals** and return `{"train_losses": list, "val_losses": list, "val_roc_auc": round4, "params": {"W1", "b1", "W2", "b2"}}`.

In [ ]:
def train_numpy_mlp_classifier(
    X_train, y_train, X_val, y_val,
    hidden_dim=16, lr=0.1, epochs=300, seed=42,
):
    """Train a He-init pure-NumPy 2-layer ReLU MLP binary classifier."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X_tr = np.asarray(X_train, dtype=float)
    y_tr = np.asarray(y_train, dtype=float).reshape(-1, 1)
    X_va = np.asarray(X_val, dtype=float)
    y_va = np.asarray(y_val, dtype=float).reshape(-1, 1)
    n, d_in = X_tr.shape

    rng = np.random.default_rng(seed)
    W1 = rng.normal(0.0, np.sqrt(2.0 / d_in), size=(d_in, hidden_dim))
    b1 = np.zeros((1, hidden_dim), dtype=float)
    W2 = rng.normal(
        0.0, np.sqrt(2.0 / hidden_dim), size=(hidden_dim, 1)
    )
    b2 = np.zeros((1, 1), dtype=float)

    def _bce(y_c, p_raw):
        pc = np.clip(p_raw, 1e-12, 1.0 - 1e-12)
        return -float(np.mean(
            y_c * np.log(pc) + (1.0 - y_c) * np.log(1.0 - pc)
        ))

    train_losses, val_losses = [], []
    for _ in range(epochs):
        Z1 = X_tr @ W1 + b1
        A1 = np.maximum(0.0, Z1)
        Z2 = A1 @ W2 + b2
        p_tr = 1.0 / (1.0 + np.exp(-np.clip(Z2, -500.0, 500.0)))
        train_losses.append(_bce(y_tr, p_tr))

        A1_v = np.maximum(0.0, X_va @ W1 + b1)
        Z2_v = A1_v @ W2 + b2
        p_va = 1.0 / (1.0 + np.exp(-np.clip(Z2_v, -500.0, 500.0)))
        val_losses.append(_bce(y_va, p_va))

        dZ2 = (p_tr - y_tr) / n
        dW2 = A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dZ1 = (dZ2 @ W2.T) * (Z1 > 0).astype(float)
        dW1 = X_tr.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        W1 = W1 - lr * dW1
        b1 = b1 - lr * db1
        W2 = W2 - lr * dW2
        b2 = b2 - lr * db2

    A1_v = np.maximum(0.0, X_va @ W1 + b1)
    p_va = 1.0 / (1.0 + np.exp(-np.clip(A1_v @ W2 + b2, -500.0, 500.0)))
    val_auc = round(float(roc_auc_score(y_va.ravel(), p_va.ravel())), 4)
    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_roc_auc": val_auc,
        "params": {"W1": W1, "b1": b1, "W2": W2, "b2": b2},
    }
    ### END SOLUTION


---
## Exam Task 27.3 (35 Points): `compare_numpy_mlp_vs_logistic_mch(mch_df, seed=42)`

Benchmark your pure-NumPy 2-layer MLP against `LogisticRegression(max_iter=500)` on `mch_survey_sample` without mutating `mch_df`:
1. Copy `mch_df`, replace `anc_visits == 999` with `np.nan`, title-case `residence` -> `is_urban`, and map `wealth_quintile` (`Poorest`..`Richest`) to ordinal `1..5` (`wealth_num`).
2. Drop rows with NaNs across `["anc_visits", "maternal_hb_g_dl", "maternal_age_years", "is_urban", "wealth_num", "child_stunted"]`.
3. Run a stratified 75/25 split (`test_size=0.25, stratify=y, random_state=seed`) and fit `StandardScaler()` strictly on `X_train`.
4. Train a pure-NumPy 2-layer ReLU MLP (`hidden_dim=16, lr=0.1, epochs=400, seed=seed`) and `LogisticRegression(max_iter=500)` on the scaled training split, returning `{"mlp_auc": round4, "logistic_auc": round4, "mlp_final_loss": round5}`.

In [ ]:
def compare_numpy_mlp_vs_logistic_mch(mch_df, seed=42):
    """Clean MCH data and benchmark NumPy MLP vs. LogisticRegression."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    work = mch_df.copy()
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    wmap = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_num"] = (
        work["wealth_quintile"].astype(str).str.strip()
        .str.title().map(wmap)
    )
    feats = [
        "anc_visits", "maternal_hb_g_dl",
        "maternal_age_years", "is_urban", "wealth_num",
    ]
    clean = work[feats + ["child_stunted"]].dropna()
    X = clean[feats].to_numpy(dtype=float)
    y = clean["child_stunted"].to_numpy(dtype=int)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=seed
    )
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    log_reg = LogisticRegression(max_iter=500)
    log_reg.fit(X_tr_s, y_tr)
    log_auc = round(float(
        roc_auc_score(y_te, log_reg.predict_proba(X_te_s)[:, 1])
    ), 4)

    rng = np.random.default_rng(seed)
    n, d_in = X_tr_s.shape
    h = 16
    W1 = rng.normal(0.0, np.sqrt(2.0 / d_in), size=(d_in, h))
    b1 = np.zeros((1, h), dtype=float)
    W2 = rng.normal(0.0, np.sqrt(2.0 / h), size=(h, 1))
    b2 = np.zeros((1, 1), dtype=float)
    y_col = y_tr.astype(float).reshape(-1, 1)
    final_loss = 0.0
    for _ in range(400):
        Z1 = X_tr_s @ W1 + b1
        A1 = np.maximum(0.0, Z1)
        Z2 = A1 @ W2 + b2
        probs = 1.0 / (1.0 + np.exp(-np.clip(Z2, -500.0, 500.0)))
        pc = np.clip(probs, 1e-12, 1.0 - 1e-12)
        final_loss = -float(np.mean(
            y_col * np.log(pc) + (1.0 - y_col) * np.log(1.0 - pc)
        ))
        dZ2 = (probs - y_col) / n
        dW2 = A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dZ1 = (dZ2 @ W2.T) * (Z1 > 0).astype(float)
        dW1 = X_tr_s.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        W1 = W1 - 0.1 * dW1
        b1 = b1 - 0.1 * db1
        W2 = W2 - 0.1 * dW2
        b2 = b2 - 0.1 * db2

    A1_te = np.maximum(0.0, X_te_s @ W1 + b1)
    p_te = 1.0 / (1.0 + np.exp(-np.clip(A1_te @ W2 + b2, -500.0, 500.0)))
    mlp_auc = round(float(roc_auc_score(y_te, p_te.ravel())), 4)
    return {
        "mlp_auc": mlp_auc,
        "logistic_auc": log_auc,
        "mlp_final_loss": round(final_loss, 5),
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch27_exam_01": solve_xor_mlp_numpy,
    "ch27_exam_02": train_numpy_mlp_classifier,
    "ch27_exam_03": compare_numpy_mlp_vs_logistic_mch,
}

tutor.check_exam("ch27", exam_submissions)


---
## Chapter 27 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 27.1 (Zero Initialization Symmetry Trap):** Initializing all hidden weights `W1 = np.zeros((2, 8))` causes every hidden neuron to receive the exact same gradient on every epoch, collapsing an 8-neuron hidden layer into a single linear unit that can never solve XOR. Always break symmetry with seeded random weights (`np.random.default_rng(seed)`).
- **Task 27.2 (He Initialization for ReLU):** Scaling `W1` by `np.sqrt(2.0 / d_in)` compensates for ReLU zeroing roughly half of all activations, preserving gradient variance across layers and driving smooth convergence on nonlinear datasets.
- **Task 27.3 (Leak-Free Scaling & Fair Baselines):** Fitting `StandardScaler` on the full dataset before `train_test_split` leaks test statistics into both the MLP and `LogisticRegression`. Always fit strictly on `X_train` (`fit_transform(X_train)`, `transform(X_test)`) and avoid `inplace=True` in Pandas 3.x.